In [ ]:
# Import necessary libraries
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline

def calc_parc_o3(var, p_top=30, p_bottom=100):
    """
    Calculate the partial ozone column between specified pressure levels.
    (Unmodified from original)
    """
    m_air = 28.964 / (6.022E23)
    g = 980.6

    if 'plev' in var.dims:
        plev = var.plev; level = 'plev'
    elif 'lev' in var.dims:
        plev = var.lev; level = 'lev'
    elif 'level' in var.dims:
        plev = var.level; level = 'level'
    else:
        raise ValueError('No pressure level coordinate found in data.')

    time = var.time
    delta_p = np.zeros((len(time), len(plev)))

    if plev[0] < plev[-1] and plev[-1] <= 1000:
        factor, factor_2 = 100, 1
    elif plev[0] > plev[-1] and plev[0] <= 1000:
        factor, factor_2 = 100, 1
    elif plev[0] < plev[-1] and plev[-1] > 1000:
        factor, factor_2 = 1, 100
    else:
        factor, factor_2 = 1, 100

    if plev[0] < plev[-1]:
        for i in range(1, len(plev)):
            delta_p[:, i] = plev[i] - plev[i-1]
        weights_p = xr.DataArray(delta_p * factor, dims=['time', level], coords=[time, plev])
        O3 = var * weights_p * 10 / (g * m_air)
        O3 = O3.sel(**{level: slice(p_top * factor_2, p_bottom * factor_2)})
        O3 = O3.sum(dim=level) / 2.687E16
    else:
        for i in range(len(plev) - 1):
            delta_p[:, i] = plev[i] - plev[i+1]
        weights_p = xr.DataArray(delta_p * factor, dims=['time', level], coords=[time, plev])
        O3 = var * weights_p * 10 / (g * m_air)
        O3 = O3.sel(**{level: slice(p_bottom * factor_2, p_top * factor_2)})
        O3 = O3.sum(dim=level) / 2.687E16

    return O3.where(O3 != 0)

def spatial_average(O3_data, lat_start=60, lat_end=90):
    """
    Compute spatial average; skip lon average if no lon dim.
    """
    if 'lon' in O3_data.dims:
        O3_avg = O3_data.mean(dim='lon')
    else:
        O3_avg = O3_data
    var = O3_avg.sel(lat=slice(lat_start, lat_end))
    weights = np.cos(np.deg2rad(var.lat))
    return var.weighted(weights).mean(dim='lat')

def load_experiment_data(file_pattern, lat_start=60, lat_end=90):
    ds = xr.open_mfdataset(
        file_pattern,
        concat_dim='member',
        combine='nested',
        data_vars=['O3'],
        coords='minimal',
        chunks={'time': 100},
        drop_variables=['PS'],
        compat='override',
        combine_attrs='override',
        cache=False,              # 关掉 netCDF4 的内部缓存
    )
    ds = ds.sel(lat=slice(lat_start, lat_end))
    o3 = ds['O3'].mean(dim='lon')
    o3_30_70 = calc_parc_o3(o3, 30, 70)
    weights = np.cos(np.deg2rad(o3_30_70.lat))
    result = o3_30_70.weighted(weights).mean(dim='lat').load()  # 立刻全量 load 到内存
    ds.close()                                              # 及时关闭所有底层文件
    return result



def load_reference_data(year):
    """
    Original reference + climatology plus tropical.
    Returns: ref_data, clim_data, ref_trop, clim_trop
    """
    file_pattern = '/mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/atm/hist/h1/O3/' + \
                   'CO2x1SmidEmin_yBWCN.cam.h1.*.O3.isobar.nc'
    ds = xr.open_mfdataset(file_pattern, combine='by_coords')
    o3 = ds['O3']
    o3_30_70 = calc_parc_o3(o3, 30, 70)
    var_all_p = spatial_average(o3_30_70, 60, 90)
    var_all_t = spatial_average(o3_30_70, -10, 10)

    merged = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc'
    ds_old = xr.open_dataset(merged)
    o3_old = ds_old['O3']
    o3_30_70_old = calc_parc_o3(o3_old, 30, 70)
    var_old_p = spatial_average(o3_30_70_old, 60, 90)
    var_old_t = spatial_average(o3_30_70_old, -10, 10)

    ref_data  = var_old_p.sel(time=var_old_p.time.dt.year == int(year))
    ref_trop  = var_old_t.sel(time=var_old_t.time.dt.year == int(year))
    clim_data = var_all_p.groupby('time.dayofyear').mean()
    clim_trop = var_all_t.groupby('time.dayofyear').mean()
    ref_data  = ref_data.load()
    clim_data = clim_data.load()
    ds_old.close()
    return ref_data, clim_data, ref_trop.load(), clim_trop.load()

def load_year_small_pert(year, lat_start=60, lat_end=90):
    base = f'/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{year}'
    feb = os.path.join(base, 'Feb', f'BWCN.e122.f19_g16.002.{year}-02.*.nc')
    mar = os.path.join(base, 'Mar', f'BWCN.e122.f19_g16.002.{year}-03.*.nc')
    return load_experiment_data(feb, lat_start, lat_end), load_experiment_data(mar, lat_start, lat_end)


def load_year_nocouple(year, lat_start=60, lat_end=90):
    feb = f'/home/weiji/restart_exam/nocouple_data/{year}/Feb_NOCOUPL/{year}-02/*.nc'
    mar = f'/home/weiji/restart_exam/nocouple_data/{year}/Mar_NOCOUPL/{year}-03/*.nc'
    return load_experiment_data(feb, lat_start, lat_end), load_experiment_data(mar, lat_start, lat_end)

def unified_plot(ref_p, clim_p, ref_t, clim_t, experiments, year,
                 pressure_range='30-70', prefix='O3_relative',
                 xlim_start=0, xlim_end=150,
                 xticks=None, xtick_labels=None):
    """
    原 signature 完全不变。
    1) 一次性算好 ref_p_rel, ref_t_rel 和各 exp 的 pol_rel/trop_rel
    2) 轻量地做 fill/plot/format/save
    """
    # ——— 1. 预计算相对百分比，只 groupby 一次 ———
    ref_p_rel = (ref_p.groupby('time.dayofyear') / clim_p) * 100
    ref_t_rel = (ref_t.groupby('time.dayofyear') / clim_t) * 100

    for exp in experiments:
        exp['pol_rel']  = (exp['pol'].groupby('time.dayofyear')  / clim_p) * 100
        exp['trop_rel'] = (exp['trop'].groupby('time.dayofyear') / clim_t) * 100

    # ——— 2. 生成 xticks/labels ———
    months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    ticks  = [0,31,59,90,120,151,181,212,243,273,304,334]
    if xticks is None:
        xticks = [t for t in ticks if xlim_start <= t <= xlim_end]
    if xtick_labels is None:
        xtick_labels = [months[i] for i,t in enumerate(ticks) if t in xticks]

    # ——— 3. 画图主体 ———
    fig, (ax_p, ax_t) = plt.subplots(2,1, sharex=True, figsize=(12,10), constrained_layout=True)

    # Reference
    x_ref = np.arange(xlim_start, min(xlim_end, ref_p_rel.sizes['time']))
    ax_p.plot(x_ref,
              ref_p_rel.isel(time=slice(xlim_start, xlim_end)),
              color='black', lw=3, label='Reference')
    ax_t.plot(x_ref,
              ref_t_rel.isel(time=slice(xlim_start, xlim_end)),
              color='black', lw=3, ls='--', label='Reference Tropics')

    # Experiments
    for exp in experiments:
        off    = exp['offset']
        lc     = exp['line_color']
        # 极区
        pol = exp['pol_rel']; Np = pol.sizes['time']
        if off < xlim_end:
            st = max(0, xlim_start - off)
            ed = min(Np,   xlim_end   - off)
            xx = np.arange(off + st, off + ed)
            seg = pol.isel(time=slice(st, ed))
            if 'member' in seg.dims:
                ax_p.fill_between(xx,
                                  seg.min('member'),
                                  seg.max('member'),
                                  color=lc, alpha=0.3)
                ax_p.plot(xx,
                          seg.mean('member'),
                          color=lc, lw=2,label=exp['label'])
            else:
                ax_p.plot(xx, seg,
                          color=lc, lw=2,
                          label=exp['label'])
        # 热带
        trop = exp['trop_rel']; Nt = trop.sizes['time']
        if off < xlim_end:
            st = max(0, xlim_start - off)
            ed = min(Nt,   xlim_end   - off)
            xx = np.arange(off + st, off + ed)
            seg = trop.isel(time=slice(st, ed))
            md  = seg.mean('member') if 'member' in seg.dims else seg
            ax_t.plot(xx, md,
                      color=lc, lw=2, ls='--',label=exp['label'])

    # ——— 4. 格式化 & 保存 ———
    ax_p.set_ylabel('Polar O3 (% of climatology)',   fontsize=14)
    ax_t.set_ylabel('Tropical O3 (% of climatology)',fontsize=14)
    ax_t.set_xlabel('Day of year',                  fontsize=14)

    ax_p.set_xticks(xticks);  ax_t.set_xticks(xticks)
    ax_t.set_xticklabels(xtick_labels, fontsize=12)
    ax_p.legend(fontsize=12);  ax_t.legend(fontsize=12)
    ax_p.set_xlim(xlim_start, xlim_end)
    ax_t.set_xlim(xlim_start, xlim_end)

    ax_p.text(0.02, 0.95, f'Year: {year}',
              transform=ax_p.transAxes,
              fontsize=14, va='top',
              bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    out = '/home/weiji/restart_exam/20250506/O3withnewclim_percent'
    os.makedirs(out, exist_ok=True)
    fname = f'{out}/{prefix}_{year}_{pressure_range.replace("-","to")}'
    fig.savefig(f'{fname}.pdf', dpi=300)
    fig.savefig(f'{fname}.png', dpi=300)
    plt.close(fig)
    for exp in experiments:
        for k in ('pol_rel','trop_rel','pol_np','trop_np'):
            if k in exp: del exp[k]
    del ref_p_rel, ref_t_rel
    import gc; gc.collect()
    return fig, (ax_p, ax_t)

# --- Data Loading and Plotting ---

# 0008 small
fJ = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Jan/BWCN.e122.f19_g16.002.0008-01.*.nc'
fF = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/BWCN.e122.f19_g16.002.0008-02.*.nc'
fM = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/BWCN.e122.f19_g16.002.0008-03.*.nc'
dJp = load_experiment_data(fJ); dJt = load_experiment_data(fJ, -10,10)
dFp = load_experiment_data(fF); dFt = load_experiment_data(fF, -10,10)
dMp = load_experiment_data(fM); dMt = load_experiment_data(fM, -10,10)

# 0008 large
fFL = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_2/BWCN.e122.f19_g16.002.0008-02.*.nc'
fML = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_2/BWCN.e122.f19_g16.002.0008-03.*.nc'
fAL = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Apr/BWCN.e122.f19_g16.002.0008-04.*.nc'
dFLp = load_experiment_data(fFL); dFLt = load_experiment_data(fFL, -10,10)
dMLp = load_experiment_data(fML); dMLt = load_experiment_data(fML, -10,10)
dALp = load_experiment_data(fAL); dALt = load_experiment_data(fAL, -10,10)

# 0008 moist
fFM = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_3/BWCN.e122.f19_g16.002.0008-02.*.nc'
fMM = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_3/BWCN.e122.f19_g16.002.0008-03.*.nc'
dFMp = load_experiment_data(fFM); dFMt = load_experiment_data(fFM, -10,10)
dMMp = load_experiment_data(fMM); dMMt = load_experiment_data(fMM, -10,10)

# 0008 nocouple
fFN = f'/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc'
fMN = f'/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc'
dFNp = load_experiment_data(fFN); dFNt = load_experiment_data(fFN, -10,10)
dMNp = load_experiment_data(fMN); dMNt = load_experiment_data(fMN, -10,10)

# references
ref_p_0008, clim_p_0008, ref_t_0008, clim_t_0008 = load_reference_data('0008')

# Figure 1
ex1 = [
    {'label':'0008Jan_small','offset':0,'pol':dJp,'trop':dJt,'line_color':'forestgreen','fill_color':'forestgreen'},
    {'label':'0008Feb_small','offset':31,'pol':dFp,'trop':dFt,'line_color':'royalblue','fill_color':'royalblue'},
    {'label':'0008Mar_small','offset':59,'pol':dMp,'trop':dMt,'line_color':'hotpink','fill_color':'hotpink'},
]
fig1,_ = unified_plot(ref_p_0008, clim_p_0008, ref_t_0008, clim_t_0008, ex1, year='0008',
                      prefix='O3_evolution_fig1'); plt.close()

# Figure 2
ex2 = [
    {'label':'0008Feb_large','offset':31,'pol':dFLp,'trop':dFLt,'line_color':'forestgreen','fill_color':'forestgreen'},
    {'label':'0008Mar_large','offset':59,'pol':dMLp,'trop':dMLt,'line_color':'royalblue','fill_color':'royalblue'},
    {'label':'0008Apr_large','offset':90,'pol':dALp,'trop':dALt,'line_color':'hotpink','fill_color':'hotpink'},
]
fig2,_ = unified_plot(ref_p_0008, clim_p_0008, ref_t_0008, clim_t_0008, ex2, year='0008',
                      prefix='O3_evolution_fig2'); plt.close()

# Figure 3
ex3 = [
    {'label':'0008Feb_small','offset':31,'pol':dFp,'trop':dFt,'line_color':'forestgreen','fill_color':'forestgreen'},
    {'label':'0008Feb_large','offset':31,'pol':dFLp,'trop':dFLt,'line_color':'royalblue','fill_color':'royalblue'},
    {'label':'0008Feb_moist','offset':31,'pol':dFMp,'trop':dFMt,'line_color':'hotpink','fill_color':'hotpink'},
]
fig3,_ = unified_plot(ref_p_0008, clim_p_0008, ref_t_0008, clim_t_0008, ex3, year='0008',
                      prefix='O3_evolution_fig3'); plt.close()

# Figure 4
ex4 = [
    {'label':'0008Mar_small','offset':59,'pol':dMp,'trop':dMt,'line_color':'forestgreen','fill_color':'forestgreen'},
    {'label':'0008Mar_large','offset':59,'pol':dMLp,'trop':dMLt,'line_color':'royalblue','fill_color':'royalblue'},
    {'label':'0008Mar_moist','offset':59,'pol':dMMp,'trop':dMMt,'line_color':'hotpink','fill_color':'hotpink'},
]
fig4,_ = unified_plot(ref_p_0008, clim_p_0008, ref_t_0008, clim_t_0008, ex4, year='0008',
                      prefix='O3_evolution_fig4'); plt.close()

# Figure 5 (0003 small)
d3Fp, d3Mp = load_year_small_pert('0003')
d3Ft, d3Mt = load_year_small_pert('0003', -10,10)

r3p, c3p, r3t, c3t = load_reference_data('0003')
ex5 = [
    {'label':'0003Feb_small','offset':31,'pol':d3Fp,'trop':d3Ft,'line_color':'forestgreen','fill_color':'forestgreen'},
    {'label':'0003Mar_small','offset':59,'pol':d3Mp,'trop':d3Mt,'line_color':'royalblue','fill_color':'royalblue'},
]
fig5,_ = unified_plot(r3p, c3p, r3t, c3t, ex5, year='0003', prefix='O3_evolution_fig5'); plt.close()

# Figure 6 (0013 small)
d13Fp, d13Mp = load_year_small_pert('0013')
d13Ft, d13Mt = load_year_small_pert('0013', -10,10)

r13p, c13p, r13t, c13t = load_reference_data('0013')
ex6 = [
    {'label':'0013Feb_small','offset':31,'pol':d13Fp,'trop':d13Ft,'line_color':'forestgreen','fill_color':'forestgreen'},
    {'label':'0013Mar_small','offset':59,'pol':d13Mp,'trop':d13Mt,'line_color':'royalblue','fill_color':'royalblue'},
]
fig6,_ = unified_plot(r13p, c13p, r13t, c13t, ex6, year='0013', prefix='O3_evolution_fig6'); plt.close()

# Figure 7 (0014 small)
d14Fp, d14Mp = load_year_small_pert('0014')
d14Ft,d14Mt = load_year_small_pert('0014', -10,10)

r14p, c14p, r14t, c14t = load_reference_data('0014')
ex7 = [
    {'label':'0014Feb_small','offset':31,'pol':d14Fp,'trop':d14Ft,'line_color':'forestgreen','fill_color':'forestgreen'},
    {'label':'0014Mar_small','offset':59,'pol':d14Mp,'trop':d14Mt,'line_color':'royalblue','fill_color':'royalblue'},
]
fig7,_ = unified_plot(r14p, c14p, r14t, c14t, ex7, year='0014', prefix='O3_evolution_fig7'); plt.close()

# Figure 8 (0019 small)
d19Fp, d19Mp = load_year_small_pert('0019')
d19Ft, d19Mt = load_year_small_pert('0019', -10,10)

r19p, c19p, r19t, c19t = load_reference_data('0019')
ex8 = [
    {'label':'0019Feb_small','offset':31,'pol':d19Fp,'trop':d19Ft,'line_color':'forestgreen','fill_color':'forestgreen'},
    {'label':'0019Mar_small','offset':59,'pol':d19Mp,'trop':d19Mt,'line_color':'royalblue','fill_color':'royalblue'},
]
fig8,_ = unified_plot(r19p, c19p, r19t, c19t, ex8, year='0019', prefix='O3_evolution_fig8'); plt.close()

# Figure 9 (0008 nocouple)
ex9 = [
    {'label':'0008Feb_nocouple','offset':31,'pol':dFNp,'trop':dFNt,'line_color':'forestgreen','fill_color':'forestgreen'},
    {'label':'0008Mar_nocouple','offset':59,'pol':dMNp,'trop':dMNt,'line_color':'royalblue','fill_color':'royalblue'},
]
fig9,_ = unified_plot(ref_p_0008, clim_p_0008, ref_t_0008, clim_t_0008, ex9, year='0008', prefix='O3_evolution_fig9'); plt.close()

# Figure 10 (0013 nocouple)
d13Fn, d13Mn = load_year_nocouple('0013'); 
d13Fnt, d13Mnt = load_year_nocouple('0013', -10,10)

ex10 = [
    {'label':'0013Feb_nocouple','offset':31,'pol':d13Fn,'trop':d13Fnt,'line_color':'forestgreen','fill_color':'forestgreen'},
    {'label':'0013Mar_nocouple','offset':59,'pol':d13Mn,'trop':d13Mnt,'line_color':'royalblue','fill_color':'royalblue'},
]
fig10,_ = unified_plot(r13p, c13p, r13t, c13t, ex10, year='0013', prefix='O3_evolution_fig10'); plt.close()

# Figure 11 (0014 nocouple)
d14Fn, d14Mn = load_year_nocouple('0014'); 
d14Fnt,d14Mnt = load_year_nocouple('0014', -10,10)
ex11 = [
    {'label':'0014Feb_nocouple','offset':31,'pol':d14Fn,'trop':d14Fnt,'line_color':'forestgreen','fill_color':'forestgreen'},
    {'label':'0014Mar_nocouple','offset':59,'pol':d14Mn,'trop':d14Mnt,'line_color':'royalblue','fill_color':'royalblue'},
]
fig11,_ = unified_plot(r14p, c14p, r14t, c14t, ex11, year='0014', prefix='O3_evolution_fig11'); plt.close()

# Figure 12 (0019 nocouple)
d19Fn, d19Mn = load_year_nocouple('0019'); 
d19Fnt,d19Mnt = load_year_nocouple('0019', -10,10)
ex12 = [
    {'label':'0019Feb_nocouple','offset':31,'pol':d19Fn,'trop':d19Fnt,'line_color':'forestgreen','fill_color':'forestgreen'},
    {'label':'0019Mar_nocouple','offset':59,'pol':d19Mn,'trop':d19Mnt,'line_color':'royalblue','fill_color':'royalblue'},
]
fig12,_ = unified_plot(r19p, c19p, r19t, c19t, ex12, year='0019', prefix='O3_evolution_fig12'); plt.close()

# Figure 13 (0008 small vs nocouple Feb)
ex13 = [
    {'label':'0008Feb_small','offset':31,'pol':dFp,'trop':dFt,'line_color':'forestgreen','fill_color':'forestgreen'},
    {'label':'0008Feb_nocouple','offset':31,'pol':dFNp,'trop':dFNt,'line_color':'royalblue','fill_color':'royalblue'},
]
fig13,_ = unified_plot(ref_p_0008, clim_p_0008, ref_t_0008, clim_t_0008, ex13, year='0008', prefix='O3_evolution_fig13'); plt.close()

# Figure 14 (0008 Mar small vs nocouple)
ex14 = [
    {'label':'0008Mar_small','offset':59,'pol':dMp,'trop':dMt,'line_color':'forestgreen','fill_color':'forestgreen'},
    {'label':'0008Mar_nocouple','offset':59,'pol':dMNp,'trop':dMNt,'line_color':'royalblue','fill_color':'royalblue'},
]
fig14,_ = unified_plot(ref_p_0008, clim_p_0008, ref_t_0008, clim_t_0008, ex14, year='0008', prefix='O3_evolution_fig14'); plt.close()

# Figure 15 (0013 Feb small vs nocouple)
ex15 = [
    {'label':'0013Feb_small','offset':31,'pol':d13Fp,'trop':d13Ft,'line_color':'forestgreen','fill_color':'forestgreen'},
    {'label':'0013Feb_nocouple','offset':31,'pol':d13Fn,'trop':d13Fnt,'line_color':'royalblue','fill_color':'royalblue'},
]
fig15,_ = unified_plot(r13p, c13p, r13t, c13t, ex15, year='0013', prefix='O3_evolution_fig15'); plt.close()

# Figure 16 (0013 Mar small vs nocouple)
ex16 = [
    {'label':'0013Mar_small','offset':59,'pol':d13Mp,'trop':d13Mt,'line_color':'forestgreen','fill_color':'forestgreen'},
    {'label':'0013Mar_nocouple','offset':59,'pol':d13Mn,'trop':d13Mnt,'line_color':'royalblue','fill_color':'royalblue'},
]
fig16,_ = unified_plot(r13p, c13p, r13t, c13t, ex16, year='0013', prefix='O3_evolution_fig16'); plt.close()

# Figure 17 (0014 Feb small vs nocouple)
ex17 = [
    {'label':'0014Feb_small','offset':31,'pol':d14Fp,'trop':d14Ft,'line_color':'forestgreen','fill_color':'forestgreen'},
    {'label':'0014Feb_nocouple','offset':31,'pol':d14Fn,'trop':d14Fnt,'line_color':'royalblue','fill_color':'royalblue'},
]
fig17,_ = unified_plot(r14p, c14p, r14t, c14t, ex17, year='0014', prefix='O3_evolution_fig17'); plt.close()

# Figure 18 (0014 Mar small vs nocouple)
ex18 = [
    {'label':'0014Mar_small','offset':59,'pol':d14Mp,'trop':d14Mt,'line_color':'forestgreen','fill_color':'forestgreen'},
    {'label':'0014Mar_nocouple','offset':59,'pol':d14Mn,'trop':d14Mnt,'line_color':'royalblue','fill_color':'royalblue'},
]
fig18,_ = unified_plot(r14p, c14p, r14t, c14t, ex18, year='0014', prefix='O3_evolution_fig18'); plt.close()

# Figure 19 (0019 Feb small vs nocouple)
ex19 = [
    {'label':'0019Feb_small','offset':31,'pol':d19Fp,'trop':d19Ft,'line_color':'forestgreen','fill_color':'forestgreen'},
    {'label':'0019Feb_nocouple','offset':31,'pol':d19Fn,'trop':d19Fnt,'line_color':'royalblue','fill_color':'royalblue'},
]
fig19,_ = unified_plot(r19p, c19p, r19t, c19t, ex19, year='0019', prefix='O3_evolution_fig19'); plt.close()

# Figure 20 (0019 Mar small vs nocouple)
ex20 = [
    {'label':'0019Mar_small','offset':59,'pol':d19Mp,'trop':d19Mt,'line_color':'forestgreen','fill_color':'forestgreen'},
    {'label':'0019Mar_nocouple','offset':59,'pol':d19Mn,'trop':d19Mnt,'line_color':'royalblue','fill_color':'royalblue'},
]
fig20,_ = unified_plot(r19p, c19p, r19t, c19t, ex20, year='0019', prefix='O3_evolution_fig20'); plt.close()